# SIH26142 — Deep Learning Super-Resolution Mapping (SRM)
### Sentinel-2 (10m) → <4m using SwinIR
### Team prototype notebook — Google Colab

**Pipeline:**
1. Environment setup (GPU check, SwinIR repo, dependencies)
2. Get pretrained SwinIR weights (baseline sanity check)
3. Build a real HR/LR paired training set for satellite imagery (this is the part most teams skip and lose marks on — a model that just looks sharp without a genuine HR reference is exactly the "hallucination risk" your own trade-off table warns about for GANs, and it silently applies to Transformers too if you skip proper supervision)
4. Fine-tune SwinIR on the paired dataset
5. Run inference on your own Sentinel-2 AOI (Copernicus Data Space)
6. Evaluate (PSNR/SSIM) + basic uncertainty estimation (Monte-Carlo dropout ensemble), since the PS explicitly requires uncertainty accounting

> Run cells top to bottom. Switch Colab runtime to **GPU (T4 is fine)**: `Runtime > Change runtime type > T4 GPU`.


## 1. Environment setup

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("CUDA available:", torch.cuda.is_available())


name, memory.total [MiB]
Tesla T4, 15360 MiB
CUDA available: True


In [2]:
# Clone the official SwinIR repo
!git clone https://github.com/JingyunLiang/SwinIR.git /content/SwinIR
%cd /content/SwinIR

# Core deps
!pip install -q timm einops rasterio scikit-image sentinelhub tifffile opencv-python-headless


Cloning into '/content/SwinIR'...
remote: Enumerating objects: 333, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 333 (delta 6), reused 2 (delta 2), pack-reused 323 (from 2)
Receiving objects: 100% (333/333), 29.84 MiB | 21.16 MiB/s, done.
Resolving deltas: 100% (119/119), done.
/content/SwinIR
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.4/240.4 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.1 MB/s eta 0:00:00


## 2. Pretrained baseline weights (sanity check only)

These are trained on natural images (DIV2K etc.), NOT satellite imagery.
We download them just to confirm the pipeline runs end-to-end before we fine-tune on real satellite pairs — do not treat this stage's output as your deliverable quality.

In [3]:
import os
os.makedirs('/content/SwinIR/model_zoo/swinir', exist_ok=True)

# Classical SR x4, medium size — good balance of quality/speed on a T4
!wget -q -O /content/SwinIR/model_zoo/swinir/001_classicalSR_DF2K_s64w8_SwinIR-M_x4.pth \
  https://github.com/JingyunLiang/SwinIR/releases/download/v0.0/001_classicalSR_DF2K_s64w8_SwinIR-M_x4.pth

print("Downloaded baseline weights.")


Downloaded baseline weights.


## 3. Building a real paired HR/LR dataset — the critical decision

You cannot fine-tune a super-resolution model on Sentinel-2 alone, because Sentinel-2 has no native <4m ground truth to learn from. You need **paired** (low-res, high-res) satellite imagery of the *same locations*.

We use **SEN2VENUS**: same-day Sentinel-2 (10m) and VENUS (5m) acquisitions across 29 global sites, loaded via the `tacoreader` library below — no manual dataset browsing required. Note this only gets you to a 2x jump (10m→5m), not your PS's full <4m target; it's the fastest way to get a working, trustworthy pipeline. For your final deliverable, swap in **WorldStrat** (Sentinel-2 vs ~1.5m commercial imagery) later — everything downstream (patch extraction, training, inference, evaluation) stays the same, you just point it at a different HR/LR source.


In [ ]:
# Load SEN2VENUS directly via tacoreader — no manual browsing/site-picking needed.
# Covers 29 locations worldwide, same-day Sentinel-2 (10m) <-> VENUS (5m) pairs.
!pip install -q tacoreader rasterio

import tacoreader.v1 as tacoreader
dataset = tacoreader.load("tacofoundation:sen2venus")

# Grab a slice of pairs for a fast prototype run (the full dataset has ~130,000 pairs).
N_PAIRS = 800
pairs = []
for i in range(0, N_PAIRS):
    row = dataset.read(i)
    lr_path, hr_path = row.read(0), row.read(1)
    pairs.append((lr_path, hr_path))
    if (i + 1) % 20 == 0:
        print(f"Loaded {i + 1}/{N_PAIRS} pairs...")

print(f"Done — loaded {len(pairs)} HR/LR pairs total")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 145.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 8.5 MB/s eta 0:00:00
Loaded 20/800 pairs...
Loaded 40/800 pairs...
Loaded 60/800 pairs...
Loaded 80/800 pairs...
Loaded 100/800 pairs...
Loaded 120/800 pairs...
Loaded 140/800 pairs...
Loaded 160/800 pairs...
Loaded 180/800 pairs...
Loaded 200/800 pairs...
Loaded 220/800 pairs...
Loaded 240/800 pairs...
Loaded 260/800 pairs...
Loaded 280/800 pairs...
Loaded 300/800 pairs...
Loaded 320/800 pairs...
Loaded 340/800 pairs...
Loaded 360/800 pairs...
Loaded 380/800 pairs...
Loaded 400/800 pairs...
Loaded 420/800 pairs...
Loaded 440/800 pairs...
Loaded 460/800 pairs...
Loaded 480/800 pairs...
Loaded 500/800 pairs...


## 4. Preprocessing: patch extraction

Once you have HR/LR pairs (either downloaded above, or your own AOI + a reference product), we tile them into aligned patches for training. SwinIR trains on fixed patch sizes (commonly 48–64px LR patches).

Adjust `SCALE` to match your actual pair (e.g. Sentinel-2 10m → VENUS 5m is x2; if you later assemble a 10m→2.5m pair for the true <4m target, set `SCALE=4` and crop to keep patch counts high).

In [ ]:
import numpy as np
import rasterio
from pathlib import Path

SCALE = 2          # set to match your actual HR:LR ratio (e.g. 2 for 10m->5m, 4 for 10m->2.5m)
LR_PATCH = 48       # LR patch size in pixels
HR_PATCH = LR_PATCH * SCALE

def read_bands(path, bands=(1,2,3)):
    """Read RGB (or chosen) bands from a GeoTIFF, normalize to [0,1] float32."""
    with rasterio.open(path) as src:
        arr = src.read(bands).astype(np.float32)
    arr = arr / (arr.max() + 1e-6)
    return np.transpose(arr, (1, 2, 0))  # H,W,C

def extract_patches(hr_img, lr_img, hr_patch, scale, stride=None):
    stride = stride or hr_patch
    lr_patch = hr_patch // scale
    pairs = []
    h, w, _ = lr_img.shape
    for y in range(0, h - lr_patch + 1, stride // scale):
        for x in range(0, w - lr_patch + 1, stride // scale):
            lr_crop = lr_img[y:y+lr_patch, x:x+lr_patch]
            hr_crop = hr_img[y*scale:y*scale+hr_patch, x*scale:x*scale+hr_patch]
            if lr_crop.shape[:2] == (lr_patch, lr_patch) and hr_crop.shape[:2] == (hr_patch, hr_patch):
                pairs.append((lr_crop, hr_crop))
    return pairs

print("Patch extraction helpers ready. Point read_bands() at your downloaded HR/LR GeoTIFFs.")


### Load each pair as real image arrays

`read_bands()` above only reads one file when called — here we run it across every pair to build two matching lists of actual pixel data.

In [ ]:
hr_list = []
lr_list = []

for lr_path, hr_path in pairs:
    lr_img = read_bands(lr_path)
    hr_img = read_bands(hr_path)
    lr_list.append(lr_img)
    hr_list.append(hr_img)

print(f"Loaded {len(lr_list)} LR images and {len(hr_list)} HR images")
print("Example LR shape:", lr_list[0].shape)
print("Example HR shape:", hr_list[0].shape)
# Sanity check: HR height/width should be exactly SCALE times the LR height/width.


### Extract training patches from every pair

Cuts each full image pair into many small aligned tiles — this is what the model actually trains on.

In [ ]:
all_patches = []

for lr_img, hr_img in zip(lr_list, hr_list):
    patches = extract_patches(hr_img, lr_img, HR_PATCH, SCALE)
    all_patches.extend(patches)

print(f"Total training patches: {len(all_patches)}")
print("Example LR patch shape:", all_patches[0][0].shape)
print("Example HR patch shape:", all_patches[0][1].shape)


### Visual sanity check

Before trusting any of this, confirm the LR/HR patches actually show the *same place* — just at different sharpness. If they look like different scenes, something misaligned upstream and training will learn garbage.

In [ ]:
import matplotlib.pyplot as plt

lr_sample, hr_sample = all_patches[0]

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(lr_sample)
ax[0].set_title("LR patch")
ax[0].axis('off')
ax[1].imshow(hr_sample)
ax[1].set_title("HR patch")
ax[1].axis('off')
plt.show()


## 5. Fine-tuning SwinIR on satellite pairs

We load the pretrained classical-SR weights as initialization (transfer learning), then fine-tune on your satellite patch pairs. This adapts the model's texture priors from natural-image statistics to satellite spectral/spatial statistics, which is the step most teams skip and then wonder why their "SR" output looks like generic photo sharpening.

In [ ]:
import sys
sys.path.append('/content/SwinIR')
from models.network_swinir import SwinIR
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = SwinIR(
    upscale=SCALE, in_chans=3, img_size=LR_PATCH, window_size=8,
    img_range=1., depths=[6,6,6,6], embed_dim=60, num_heads=[6,6,6,6],
    mlp_ratio=2, upsampler='pixelshuffledirect', resi_connection='1conv'
).to(device)

# Load pretrained weights as a starting point where shapes match (classical-SR checkpoint uses a larger config
# by default; for a quick prototype we train this smaller SwinIR-lite from scratch on satellite data instead
# of forcing a mismatched shape load — swap to the full 001_classicalSR config above if you want to load pretrained
# weights exactly, at the cost of more GPU memory).

class PatchDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        lr, hr = self.pairs[idx]
        lr_t = torch.from_numpy(lr.transpose(2,0,1)).float()
        hr_t = torch.from_numpy(hr.transpose(2,0,1)).float()
        return lr_t, hr_t

train_loader = DataLoader(PatchDataset(all_patches), batch_size=8, shuffle=True)

criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=2e-4)

def train_one_epoch(loader):
    model.train()
    total_loss = 0
    for lr, hr in loader:
        lr, hr = lr.to(device), hr.to(device)
        optimizer.zero_grad()
        sr = model(lr)
        loss = criterion(sr, hr)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

print("Model + training loop ready. Uncomment `pairs`/`train_loader` once patches are extracted, then call train_one_epoch() in a loop.")


In [ ]:
N_EPOCHS = 20
for epoch in range(N_EPOCHS):
    loss = train_one_epoch(train_loader)
    print(f"Epoch {epoch+1}/{N_EPOCHS} - loss: {loss:.4f}")

# Save weights so a Colab disconnect doesn't cost you the training run
torch.save(model.state_dict(), '/content/swinir_finetuned.pth')
print("Saved fine-tuned weights to /content/swinir_finetuned.pth")


## 6. Inference on your own Sentinel-2 AOI

Download an AOI from the Copernicus Data Space Browser (link in your PS) as a GeoTIFF, then run it through the fine-tuned model.

In [ ]:
def run_inference(model, lr_geotiff_path, out_path, patch=48, overlap=8):
    """Sliding-window SR inference with simple blending at patch borders."""
    model.eval()
    with rasterio.open(lr_geotiff_path) as src:
        profile = src.profile.copy()
        img = src.read([1,2,3]).astype(np.float32)
        img = img / (img.max() + 1e-6)

    c, h, w = img.shape
    out = np.zeros((c, h*SCALE, w*SCALE), dtype=np.float32)
    weight = np.zeros((h*SCALE, w*SCALE), dtype=np.float32)
    step = patch - overlap

    with torch.no_grad():
        for y in range(0, h - patch + 1, step):
            for x in range(0, w - patch + 1, step):
                crop = img[:, y:y+patch, x:x+patch]
                t = torch.from_numpy(crop).unsqueeze(0).float().to(device)
                sr = model(t).squeeze(0).cpu().numpy()
                out[:, y*SCALE:(y+patch)*SCALE, x*SCALE:(x+patch)*SCALE] += sr
                weight[y*SCALE:(y+patch)*SCALE, x*SCALE:(x+patch)*SCALE] += 1

    weight[weight == 0] = 1
    out = out / weight
    profile.update(height=out.shape[1], width=out.shape[2],
                    transform=src.transform * src.transform.scale(1/SCALE, 1/SCALE))
    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write((out * 255).clip(0,255).astype('uint8'))
    return out

print("Inference function ready. Call run_inference(model, 'your_sentinel2_tile.tif', 'sr_output.tif').")


## 7. Evaluation + uncertainty estimation

The PS explicitly requires accounting for uncertainty since reconstructed detail is *inferred*, not observed. A lightweight approach for a hackathon timeline: **Monte-Carlo dropout** — run inference N times with dropout active, and use the pixel-wise standard deviation across runs as an uncertainty map. High-variance regions = the model is less confident, which is honest and demoable.

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim

def evaluate(sr, hr_ref):
    sr_img = np.clip(sr.transpose(1,2,0), 0, 1)
    hr_img = np.clip(hr_ref.transpose(1,2,0), 0, 1)
    p = psnr(hr_img, sr_img, data_range=1.0)
    s = ssim(hr_img, sr_img, channel_axis=-1, data_range=1.0)
    return {"PSNR": p, "SSIM": s}

def mc_dropout_uncertainty(model, lr_tensor, n_passes=10):
    """Enable dropout at inference and sample repeatedly to get an uncertainty map."""
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()  # keep dropout active
    preds = []
    with torch.no_grad():
        for _ in range(n_passes):
            preds.append(model(lr_tensor).cpu().numpy())
    preds = np.stack(preds, axis=0)
    mean_pred = preds.mean(axis=0)
    uncertainty_map = preds.std(axis=0)
    return mean_pred, uncertainty_map

print("Evaluation + uncertainty helpers ready.")


## Notes for your team

- **This SwinIR config (SwinIR-lite, trained from scratch) is deliberately small** so it fits comfortably in free Colab's ~15GB T4 memory and trains fast enough for iteration during the hackathon. If you have Colab Pro or move to a bigger GPU, load the actual `001_classicalSR_DF2K_s64w8_SwinIR-M_x4.pth` weights properly (match `embed_dim=180`, `depths=[6]*6`, `num_heads=[6]*6` per the SwinIR repo's model configs) for a stronger starting point.
- **Dataset choice drives your credibility score.** Judges will likely ask what your HR ground truth was — "we downsampled Sentinel-2 and upsampled it back" is a weak answer; "we trained against VENUS/SPOT reference imagery of the same AOIs" is a strong one.
- **Report PSNR/SSIM against held-out HR tiles**, not against your own training tiles — and show the uncertainty map alongside your sharpest output image in the demo, since that directly answers the PS's "faithfulness/trustworthiness" requirement.
- Next steps once this runs end-to-end: swap in the full paired dataset, add spectral bands beyond RGB (NIR is valuable for crop monitoring per the PS use case), and consider band-wise loss weighting.
